# BenchmarkResults Filtering Workflow (Production-Oriented)

This notebook demonstrates the updated `filter_results` behavior:

- Filtering is **global across all submissions** in `openfe_benchmarks/results`.
- Return values are `BenchmarkResults` objects for matching submissions.
- Entry-level filters (for `dg`/`ddg`) still work, but selection happens at submission level.
- When using `load_results=False`, you can load a specific submission later with `load_raw_results()`.

The workflow below uses practical checks that are useful for CI and analysis pipelines, with assertions that fail fast when behavior changes.

In [ ]:
from pathlib import Path

from cinnabar import plotting

from openfe_benchmarks.results._benchmark_results import filter_results


def submission_ids(items):
    return {item.submission_id for item in items}


def flatten(items):
    for item in items:
        if isinstance(item, list):
            yield from flatten(item)
        else:
            yield item


def get_nested(obj, path):
    value = obj
    for part in path.split("__"):
        if isinstance(value, dict):
            value = value.get(part)
        elif isinstance(value, list):
            next_values = []
            for entry in value:
                if isinstance(entry, dict):
                    nested = entry.get(part)
                else:
                    nested = getattr(entry, part, None)
                if nested is not None:
                    next_values.append(nested)
            value = next_values if next_values else None
        else:
            value = getattr(value, part, None)
        if value is None:
            return None
    return value

In [ ]:
all_submissions = filter_results(load_results=False)

print(f"Total submissions discovered: {len(all_submissions)}")
print("Sample IDs:\n   ", "\n    ".join(sorted(submission_ids(all_submissions))))

assert all_submissions, "Expected at least one submission in results directory"
assert all(hasattr(r, "submission_id") for r in all_submissions)

# With load_results=False, only metadata is loaded initially.
example_submission = all_submissions[0]
print(f"Before load_raw_results: raw_results is {example_submission.raw_results}")

# Load computational results later, only when needed.
example_submission.load_raw_results()
print(
    "After load_raw_results: raw_results loaded =",
    example_submission.raw_results is not None,
)

assert example_submission.raw_results is not None

## 1) Production Metadata Filtering

These are fast, metadata-only queries (`load_results=False`) suitable for dashboards, CI checks, and release reports.

In [ ]:
rbfe_submissions = filter_results(calculation_type="rbfe", load_results=False)
recent_submissions = filter_results(date=">=2026-01-01", load_results=False)
older_submissions = filter_results(date="<=2026-08-01", load_results=False)
newer_openfe_submissions = filter_results(openfe_version=">1.8.0", load_results=False)
non_test_submissions = filter_results(
    exclude_tags=["pontibus"], tags_mode="all", load_results=False
)

print(f"RBFE submissions: {len(rbfe_submissions)}")
print(f"Submissions since 2026-01-01: {len(recent_submissions)}")
print(f"Submissions on/before 2026-08-01: {len(older_submissions)}")
print(f"Submissions with openfe_version > 2.3.0: {len(newer_openfe_submissions)}")
print(f"Non-pontibus submissions: {len(non_test_submissions)}")

assert len(rbfe_submissions) <= len(all_submissions)
assert len(recent_submissions) <= len(all_submissions)
assert len(older_submissions) <= len(all_submissions)
assert len(newer_openfe_submissions) <= len(all_submissions)
assert len(non_test_submissions) <= len(all_submissions)

## 2) Entry-Level Filtering, Submission-Level Return

These queries inspect `dg`/`ddg` entries but return whole `BenchmarkResults` submissions. This is useful in production when downstream steps need both metadata and raw result context.

In [ ]:
tyk2_related = filter_results(system_name="tyk2", load_results=True)

print(f"Submissions containing tyk2-like system names: {len(tyk2_related)}")
print("Example IDs:", sorted(submission_ids(tyk2_related))[:5])

assert isinstance(tyk2_related, list)
assert all(hasattr(r, "raw_results") for r in tyk2_related)

## 3) Double-Underscore Filtering Demonstration

- A query with a real nested value returns submissions that actually exist.
- A query with an impossible nested value returns no submissions.

This is the production behavior you want when building search/reporting workflows.

In [ ]:
# Positive case: concrete nested value that should exist.
chosen_key = "protocol_settings__protocol"
chosen_value = "RelativeHybridTopologyProtocol"
positive_result = filter_results(load_results=False, **{chosen_key: chosen_value})
positive_ids = submission_ids(positive_result)

# Negative case: concrete nested value that should not exist.
negative_result = filter_results(
    load_results=False, protocol_settings__protocol="ProtocolThatDoesNotExist"
)
negative_ids = submission_ids(negative_result)

print(f"Nested key used: {chosen_key}")
print(f"Existing value: {chosen_value}")
print(f"Existing-value matches: {len(positive_ids)}")
print("Sample matched IDs:", sorted(positive_ids)[:5])
print(f"Impossible-value matches: {len(negative_ids)}")

assert len(positive_ids) > 0, "Expected nested query to return existing submissions"
assert len(negative_ids) == 0, (
    "Expected impossible nested query to return no submissions"
)

## 4) DG/DDG to Cinnabar Plot Demonstration

These examples show how filtered submissions feed directly into cinnabar plotting:

- DG plotting from ASFE submissions via `results.dg_femaps()` and `plotting.plot_DGs`
- DDG plotting from RBFE submissions via `results.ddg_femaps()` and `plotting.plot_DDGs`

Plots are written to a local `outputs/` directory.

### 4.1 Understanding the `source` Parameter

Both `dg_femaps()` and `ddg_femaps()` accept an optional `source` parameter that controls the source identifier in the FEMap data.

**Default behavior** (when `source=None`):
- The source defaults to the submission's `submission_id`
- This is useful for tracking which submission/methodology produced each result

**Custom source** (when `source="CustomName"`):
- Override the default identifier with a custom string
- Useful when plotting multiple submissions together or comparing different naming schemes
- Must match the `source` values expected by cinnabar plotting functions

Example usage:
```python
# Uses default: source will be the submission_id
femaps = results.dg_femaps()

# Custom source: override to use a specific identifier
femaps = results.dg_femaps(source="Computational")
```

### 4.2 DG Plot Example (ASFE)

Use a real ASFE submission and plot computed vs experimental DG values with cinnabar.

We show both:
- Using the **default source** (submission_id)
- **Overriding source** to customize the plot label

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

# Load ASFE submission for Example 1 (default source)
asfe_submission_1 = filter_results(calculation_type="asfe", load_results=True)[0]

# Example 1: Using default source (submission_id)
(system_group, system_name), dg_femap_default = next(
    iter(asfe_submission_1.dg_femaps().items())
)

# Extract the actual source from the FEMap data
actual_source = dg_femap_default.source_data["source"].iloc[0]
print(f"Default source in FEMap: {actual_source}")

dg_plot_file_default = (
    output_dir
    / f"{asfe_submission_1.submission_id}_{system_group}_{system_name}_DG_default.png"
)
plotting.plot_DGs(
    dg_femap_default,
    source=actual_source,  # Use the actual source from the data
    title=f"{asfe_submission_1.submission_id} (default source)",
    figsize=5,
    scatter_kwargs={"s": 20, "marker": "o"},
    filename=dg_plot_file_default.as_posix(),
)

print(f"ASFE submission: {asfe_submission_1.submission_id}")
print(f"System: {system_group}/{system_name}")
print(f"Saved DG plot (default source): {dg_plot_file_default}")
print()

# Load a fresh ASFE submission for Example 2 (custom source)
# This is needed because FEMap results are cached after first access
asfe_submission_2 = filter_results(calculation_type="asfe", load_results=True)[0]

# Example 2: Using custom source override
custom_source_label = "OpenFE-Benchmark-Suite"
dg_femap_custom = asfe_submission_2.dg_femaps(source=custom_source_label)
(system_group, system_name), dg_femap_custom = next(iter(dg_femap_custom.items()))

custom_actual_source = dg_femap_custom.source_data["source"].iloc[0]
print(f"Custom source in FEMap: {custom_actual_source}")

dg_plot_file_custom = (
    output_dir
    / f"{asfe_submission_2.submission_id}_{system_group}_{system_name}_DG_custom.png"
)
plotting.plot_DGs(
    dg_femap_custom,
    source=custom_actual_source,  # Use the actual custom source from the data
    title=f"{asfe_submission_2.submission_id} (custom source)",
    figsize=5,
    scatter_kwargs={"s": 20, "marker": "o"},
    filename=dg_plot_file_custom.as_posix(),
)

print(f"Saved DG plot (custom source): {dg_plot_file_custom}")

### 4.3 DDG Plot Example (RBFE)

Use a real RBFE submission and plot computed vs experimental DDG values with cinnabar.

Demonstrates extracting available sources from the data and both default and custom source behavior.

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

rbfe_submission = filter_results(calculation_type="rbfe", load_results=True)[0]

# Example 1: Using default source (submission_id)
ddg_femap_default = rbfe_submission.ddg_femaps()  # source defaults to submission_id
(system_group, system_name), ddg_femap = next(iter(ddg_femap_default.items()))

relative_df = ddg_femap.get_relative_dataframe()
available_sources = sorted(set(relative_df["source"].fillna("")))
print(f"Available sources in data: {available_sources}")
print(f"Default source (submission_id): {rbfe_submission.submission_id}")

ddg_plot_file_default = (
    output_dir
    / f"{rbfe_submission.submission_id}_{system_group}_{system_name}_DDG_default.png"
)
plotting.plot_DDGs(
    ddg_femap,
    source=rbfe_submission.submission_id,  # Use the default submission_id
    title=f"{rbfe_submission.submission_id} (default)",
    figsize=5,
    scatter_kwargs={"s": 20, "marker": "o"},
    filename=ddg_plot_file_default.as_posix(),
)

print(f"\nRBFE submission: {rbfe_submission.submission_id}")
print(f"System: {system_group}/{system_name}")
print(f"Saved DDG plot (default source): {ddg_plot_file_default}")
print()

# Example 2: Using custom source override
custom_source_label = "OpenFE-Suite-v1.0"
ddg_femap_custom = rbfe_submission.ddg_femaps(source=custom_source_label)
(system_group, system_name), ddg_femap_custom = next(iter(ddg_femap_custom.items()))

ddg_plot_file_custom = (
    output_dir
    / f"{rbfe_submission.submission_id}_{system_group}_{system_name}_DDG_custom.png"
)
plotting.plot_DDGs(
    ddg_femap_custom,
    source=custom_source_label,
    title=f"{rbfe_submission.submission_id} (custom source)",
    figsize=5,
    scatter_kwargs={"s": 20, "marker": "o"},
    filename=ddg_plot_file_custom.as_posix(),
)

print(f"Saved DDG plot (custom source): {ddg_plot_file_custom}")